https://github.com/ThanhTruc23/BAI_TAP_TRI_TUE_NHAN_TAO
1.---Mô hình PEAS---
Thành phần	Nội dung
P – Performance Measure	Hoàn thành đúng trạng thái đích, ít bước di chuyển, thời gian xử lý nhanh
E – Environment:Bàn cờ 3x3 gồm 8 ô số và 1 ô trống
A – Actuators: Di chuyển ô trống lên, xuống, trái, phải
S – Sensors	Quan sát vị trí các ô số và ô trống trên bàn cờ
2.---Các thành phần của Agent theo kiến trúc Model-Based---
State (Trạng thái hiện tại)
- Agent nhận percept từ môi trường dưới dạng ma trận 3x3 của bài toán 8-Puzzle.
- State biểu diễn vị trí hiện tại của các ô số và ô trống tại thời điểm agent quan sát môi trường.
Model (Mô hình nội tại)
Agent sử dụng model để mô phỏng và cập nhật môi trường sau mỗi lần di chuyển.
Hiểu biết về môi trường
- Biết rằng ô trống chỉ có thể di chuyển:
  - UP
  - DOWN
  - LEFT
  - RIGHT
- Biết quy luật hoán đổi vị trí giữa ô trống và ô số lân cận.
- Biết giới hạn biên của ma trận 3x3 để tránh hành động không hợp lệ.

Lưu trữ thông tin (Internal State)
- Lưu trạng thái đích (Goal State).
- Lưu trạng thái hiện tại của puzzle.
- Lưu các trạng thái đã đi qua (Visited States).
- Lưu heuristic Manhattan Distance để đánh giá trạng thái.
- Lưu Plan là chuỗi các bước đi tối ưu chưa thực hiện.

4 --Rules (Tập luật & Lập luận)--
 Rule 1
IF state hiện tại == goalTHEN ACTION = STOP
 Rule 2
IF chưa đạt goal AND Plan đang rỗng
THEN:
- Kích hoạt thuật toán A*
- Tính heuristic Manhattan Distance
- Sinh chuỗi hành động tối ưu
- Lưu các bước đi vào Plan
- ACTION = Plan.pop(0)
 Rule 3
IF Plan đã có sẵn hành động THEN ACTION = Plan.pop(0)
Action (Hành động)

Các hành động mà agent có thể thực hiện:
- UP
- DOWN
- LEFT
- RIGHT
- STOP

Sau khi ACTION được thực hiện:
- State sẽ được cập nhật.
- Model nội tại cũng được cập nhật theo trạng thái mới của môi trường.

Trạng thái môi trường

GOAL STATE (Trạng thái đích)
+------+------+------+
|  1   |  2   |  3   |
+------+------+------+
|  4   |  5   |  6   |
+------+------+------+
|  7   |  8   |      |
+------+------+------+
0 hoặc ô trống = blank tile

In [1]:
import copy
import heapq

# =====================================================================
# 1. THUẬT TOÁN TÌM ĐƯỜNG A* (DÙNG HÀM ĐÁNH GIÁ: SỐ Ô LỆCH VỊ TRÍ)
# =====================================================================
class SearchNode:
    def __init__(self, board_state, parent_node, move_name, steps_taken, expected_left):
        self.board_state = board_state
        self.parent_node = parent_node
        self.move_name = move_name
        self.steps_taken = steps_taken       # Chi phí thực tế đã đi
        self.expected_left = expected_left   # Dự đoán số bước còn lại
        self.priority_score = steps_taken + expected_left

    def __lt__(self, other):
        return self.priority_score < other.priority_score

def locate_empty_space(board):
    for r in range(3):
        for c in range(3):
            if board[r][c] == 0:
                return r, c

def count_wrong_positions(current_board, target_board):
    errors = 0
    for i in range(3):
        for j in range(3):
            val = current_board[i][j]
            if val != 0 and val != target_board[i][j]:
                errors += 1
    return errors

def get_possible_moves(board):
    branches = []
    r, c = locate_empty_space(board)
    directions = [('UP', -1, 0), ('DOWN', 1, 0), ('LEFT', 0, -1), ('RIGHT', 0, 1)]
    
    for move_action, dr, dc in directions:
        new_r, new_c = r + dr, c + dc
        if 0 <= new_r <= 2 and 0 <= new_c <= 2:
            next_board = copy.deepcopy(board)
            next_board[r][c], next_board[new_r][new_c] = next_board[new_r][new_c], next_board[r][c]
            branches.append((next_board, move_action))
    return branches

def execute_a_star(start_board, target_board):
    start_errors = count_wrong_positions(start_board, target_board)
    initial_node = SearchNode(start_board, None, None, 0, start_errors)
    
    open_list = []
    heapq.heappush(open_list, initial_node)
    visited_states = set()

    while open_list:
        current_node = heapq.heappop(open_list)
        board_string = str(current_node.board_state)

        if current_node.board_state == target_board:
            path = []
            while current_node.parent_node:
                path.append(current_node.move_name)
                current_node = current_node.parent_node
            path.reverse()
            return path

        visited_states.add(board_string)

        for next_board, move_action in get_possible_moves(current_node.board_state):
            if str(next_board) in visited_states:
                continue
                
            new_steps = current_node.steps_taken + 1
            future_errors = count_wrong_positions(next_board, target_board)
            new_node = SearchNode(next_board, current_node, move_action, new_steps, future_errors)
            heapq.heappush(open_list, new_node)
            
    return []

# =====================================================================
# 2. ĐỊNH NGHĨA AGENT
# =====================================================================
class SmartReflexAgent:
    def __init__(self, final_goal):
        self.current_vision = None
        self.chosen_action = None
        self.memory = {
            'goal': final_goal,
            'action_list': [],
            'calc_count': 0
        }

    def process_and_act(self, percept):
        self.current_vision = percept
        
        if self.current_vision == self.memory['goal']:
            return 'STOP'
            
        if not self.memory['action_list']:
            self.memory['action_list'] = execute_a_star(self.current_vision, self.memory['goal'])
            self.memory['calc_count'] += 1
            
        if self.memory['action_list']:
            self.chosen_action = self.memory['action_list'].pop(0)
        else:
            self.chosen_action = 'STOP' 
            
        return self.chosen_action

# =====================================================================
# 3. CHẠY THỬ NGHIỆM
# =====================================================================
def render_board(current_board, previous_board=None):
    changed_cells = []
    if previous_board:
        for r in range(3):
            for c in range(3):
                if current_board[r][c] != 0 and current_board[r][c] != previous_board[r][c]:
                    changed_cells.append((r, c))

    print("+" + "-"*5 + "+" + "-"*5 + "+" + "-"*5 + "+")
    for r in range(3):
        row_display = "|"
        for c in range(3):
            number = current_board[r][c]
            if number == 0:
                row_display += "     |"
            elif (r, c) in changed_cells:
                row_display += f" [{number}] |"
            else:
                row_display += f"  {number}  |"
        print(row_display)
        print("+" + "-"*5 + "+" + "-"*5 + "+" + "-"*5 + "+")

target_matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

start_matrix = [
    [1, 2, 3],
    [5, 6, 0],
    [7, 8, 4]
]

bot = SmartReflexAgent(target_matrix)
current_state = copy.deepcopy(start_matrix)
old_state = None
turn_counter = 0

print("=========================================================")
print(f"TRẠNG THÁI BAN ĐẦU (Step: {turn_counter} | Số ô sai vị trí: {count_wrong_positions(current_state, target_matrix)})")
print("=========================================================")
render_board(current_state)

while True:
    turn_counter += 1
    move = bot.process_and_act(current_state)
    
    if move == 'STOP':
        print("\n=========================================================")
        print(f"KẾT QUẢ: ĐÃ GIẢI XONG 8 PUZZLE TRONG {turn_counter - 1} BƯỚC!")
        print(f"Số lần Agent phải tính toán đường đi: {bot.memory['calc_count']}")
        print("=========================================================")
        break
        
    old_state = copy.deepcopy(current_state)
    blank_r, blank_c = locate_empty_space(current_state)
    
    if move == 'UP': 
        current_state[blank_r][blank_c], current_state[blank_r-1][blank_c] = current_state[blank_r-1][blank_c], current_state[blank_r][blank_c]
    elif move == 'DOWN': 
        current_state[blank_r][blank_c], current_state[blank_r+1][blank_c] = current_state[blank_r+1][blank_c], current_state[blank_r][blank_c]
    elif move == 'LEFT': 
        current_state[blank_r][blank_c], current_state[blank_r][blank_c-1] = current_state[blank_r][blank_c-1], current_state[blank_r][blank_c]
    elif move == 'RIGHT': 
        current_state[blank_r][blank_c], current_state[blank_r][blank_c+1] = current_state[blank_r][blank_c+1], current_state[blank_r][blank_c]
    
    print("\n---------------------------------------------------------")
    print(f"Step: {turn_counter:02d} | Action: {move} | Số ô sai vị trí: {count_wrong_positions(current_state, target_matrix)}")
    render_board(current_state, old_state)

TRẠNG THÁI BAN ĐẦU (Step: 0 | Số ô sai vị trí: 3)
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
|  5  |  6  |     |
+-----+-----+-----+
|  7  |  8  |  4  |
+-----+-----+-----+

---------------------------------------------------------
Step: 01 | Action: LEFT | Số ô sai vị trí: 2
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
|  5  |     | [6] |
+-----+-----+-----+
|  7  |  8  |  4  |
+-----+-----+-----+

---------------------------------------------------------
Step: 02 | Action: LEFT | Số ô sai vị trí: 1
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
|     | [5] |  6  |
+-----+-----+-----+
|  7  |  8  |  4  |
+-----+-----+-----+

---------------------------------------------------------
Step: 03 | Action: DOWN | Số ô sai vị trí: 2
+-----+-----+-----+
|  1  |  2  |  3  |
+-----+-----+-----+
| [7] |  5  |  6  |
+-----+-----+-----+
|     |  8  |  4  |
+-----+-----+-----+

---------------------------------------------------------
Step: 04 | Action: 